In [1]:
# TODO: generalize samplers (torch, lightning, transformers?)
# TODO: analyze dataset
# TODO: test litmodels
# TODO: add wandb support
# TODO: does it do BPTT?

# Mamba with Minimal Boilerplate: Focus on the Model

In [3]:
# Auto-reload modules when they change (useful for development)
%load_ext autoreload
%autoreload 2

In [2]:
CONFIG = {
    # Reproducibility
    "seed": 42,  # Random seed for reproducibility

    # Hardware (auto-detected)
    "device": "auto",  # Auto-detect best device (CUDA/MPS/CPU)
    "precision": "auto",  # Auto-detect precision (no AMP on MPS, bf16/16 on CUDA)

    # Data
    "dataset": "fineweb-edu",  # "lotr" or "fineweb-edu"
    "dataset_subset_size": "all",  # Number of documents to load (None or "all" for streaming full dataset)
    "vocab_subset_size": 10000,  # Number of documents to build vocabulary from (only for char tokenizer)
    "block_size": 1024,  # Sequence length for training (GPT-2 tokens are more meaningful than chars)
    "train_split": 0.8,  # Fraction of data for training

    # Tokenizer
    "tokenizer": "gpt2",  # "char" for character-level, "gpt2" for GPT-2 BPE tokenizer (RECOMMENDED)

    # DataLoader (optimized for 19GB/24GB VRAM usage with streaming)
    "num_workers": 4,  # Increased from 2 (14GB/28GB RAM usage is low, can afford more workers)
    "persistent_workers": True,  # Enabled to eliminate worker spawn overhead between epochs
    "prefetch_factor": 4,  # Increased from 2 for more aggressive prefetching (better GPU saturation)
    "pin_memory": True,  # Keep enabled for CUDA performance (minor memory cost, major speed benefit)

    # Model architecture (RTX 4090 optimized with GPT-2 tokenizer)
    # GPT-2 vocab (50,257) is ~500x larger than char vocab (~100)
    # Scaled up from char-level config to utilize RTX 4090's 24GB VRAM
    "n_embed": 512,  # Embedding dimension (256→512 for richer representations)
    "hidden_size": 512,  # Hidden state dimension (256→512 for more capacity)
    "state_size": 32,  # SSM state dimension (16→32 for larger state space)
    "n_layers": 12,  # Number of Mamba layers (6→12 for deeper model)
    "expand_factor": 2,  # Internal expansion factor (unchanged)
    "conv_kernel": 4,  # Convolution kernel size (unchanged)

    # SSM Implementation
    "ssm_backend": "cuda",  # "python" or "cuda" - CUDA is 30-100x faster

    # Training (optimized for 19GB/24GB VRAM usage)
    "batch_size": 24,  # Increased from 16 to maximize GPU utilization (~1.19GB/batch → ~23GB total)
    "max_epochs": 1,  # Training epochs
    "learning_rate": 3e-4,  # Optimizer learning rate (standard for transformers)
    "lr_scheduler": "cosine",  # Cosine annealing learning rate schedule
    "early_stop_patience": 10,  # Higher patience for longer training

    # Optimization (RTX 4090 throughput improvements)
    "torch_compile": "max-autotune",  # False/None to disable, or "default"/"reduce-overhead"/"max-autotune" (30-40% speedup)
    "use_fused_adamw": True,  # Use fused AdamW for ~5-10% speedup on CUDA

    # Checkpointing
    "checkpoint_dir": "checkpoints",  # Directory to save checkpoints
    "resume_from_checkpoint": None,  # Path to checkpoint to resume from (None = train from scratch)
    
    # LitModels Upload (requires Lightning account)
    "upload_to_litmodels": False,  # Whether to upload best checkpoint to Lightning model registry
    "litmodels_name": None,  # Model name in format 'username/teamspace/modelname' (e.g., 'user/default/mamba-fineweb')
}

In [ ]:
from aiml_notebooks.hardware import init_training

# Initialize training environment and resolve auto fields
CONFIG = init_training(CONFIG, verbose=True)
CONFIG

In [4]:
from typing import List, Union

class CharacterTokenizer:
    """
    Character-level tokenizer with transformers-compatible interface.
    
    This class provides the same interface as HuggingFace tokenizers,
    making it easy to swap with any PreTrainedTokenizer.
    
    Includes UNK token support to handle unseen characters gracefully.
    """
    
    def __init__(self, text: str):
        # Build vocabulary from text
        self.vocab = sorted(list(set(text)))
        
        # Add UNK token if not present
        self.unk_token = "<UNK>"
        if self.unk_token not in self.vocab:
            self.vocab.append(self.unk_token)
        
        self.vocab_size = len(self.vocab)
        
        # Create mappings
        self._char_to_id = {c: i for i, c in enumerate(self.vocab)}
        self._id_to_char = {i: c for i, c in enumerate(self.vocab)}
        self.unk_id = self._char_to_id[self.unk_token]
    
    def encode(self, text: str) -> List[int]:
        """Convert text to token IDs. Unknown characters map to UNK token."""
        return [self._char_to_id.get(c, self.unk_id) for c in text]
    
    def decode(self, token_ids: Union[List[int], int]) -> str:
        """Convert token IDs back to text."""
        if isinstance(token_ids, int):
            return self._id_to_char.get(token_ids, self.unk_token)
        return "".join([self._id_to_char.get(i, self.unk_token) for i in token_ids])
    
    def __call__(self, text: str, **kwargs) -> dict:
        """Tokenize text (transformers-compatible interface)."""
        return {"input_ids": self.encode(text)}
    
    def __len__(self) -> int:
        return self.vocab_size


class TiktokenWrapper:
    """
    Wrapper around tiktoken to provide CharacterTokenizer-compatible interface.
    
    Uses tiktoken for fast BPE tokenization (GPT-2, GPT-3.5, GPT-4).
    """
    
    def __init__(self, model_name: str = "gpt2"):
        import tiktoken
        self.enc = tiktoken.get_encoding(model_name)
        self.vocab_size = self.enc.n_vocab
        self.model_name = model_name
    
    def encode(self, text: str) -> List[int]:
        """Convert text to token IDs."""
        return self.enc.encode(text, allowed_special="all")
    
    def decode(self, token_ids: Union[List[int], int]) -> str:
        """Convert token IDs back to text."""
        if isinstance(token_ids, int):
            token_ids = [token_ids]
        return self.enc.decode(token_ids)
    
    def __call__(self, text: str, **kwargs) -> dict:
        """Tokenize text (transformers-compatible interface)."""
        return {"input_ids": self.encode(text)}
    
    def __len__(self) -> int:
        return self.vocab_size


# Load dataset text for character tokenizer vocabulary
if CONFIG["dataset"] == "lotr":
    with open("data/lotr.txt", "r", encoding="utf-8") as f:
        text = f.read()
    dataset_text = text
    print(f"Loaded LOTR dataset: {len(text):,} characters")
    
elif CONFIG["dataset"] == "fineweb-edu":
    from datasets import load_dataset
    from itertools import islice
    
    # Get configuration
    subset_size = CONFIG.get("dataset_subset_size", 10000)
    vocab_size = CONFIG.get("vocab_subset_size", 10000)
    use_streaming = subset_size is None or subset_size == "all"
    
    # For character tokenizer: build vocabulary from a subset
    if CONFIG["tokenizer"] == "char":
        print(f"Building character vocabulary from first {vocab_size:,} documents...")
        ds_stream = load_dataset(
            "HuggingFaceFW/fineweb-edu", 
            name="sample-10BT",
            split="train",
            streaming=True
        )
        vocab_texts = [doc["text"] for doc in islice(ds_stream, vocab_size)]
        vocab_text = "\n\n".join(vocab_texts)
        print(f"Vocabulary built from {len(vocab_text):,} characters")
    else:
        vocab_text = ""  # Not needed for BPE tokenizers
    
    # Store whether we're using streaming mode
    CONFIG["_use_streaming"] = use_streaming
    
    if not use_streaming:
        # Load subset into memory (original behavior)
        print(f"Loading {subset_size:,} documents into memory...")
        ds_stream = load_dataset(
            "HuggingFaceFW/fineweb-edu", 
            name="sample-10BT",
            split="train",
            streaming=True
        )
        texts = [doc["text"] for doc in islice(ds_stream, subset_size)]
        text = "\n\n".join(texts)
        print(f"Loaded fineweb-edu: {len(text):,} characters from {len(texts):,} documents")
        dataset_text = text
    else:
        # Streaming mode - we'll tokenize on-the-fly later
        print(f"Using streaming mode - will tokenize on-the-fly")
        dataset_text = vocab_text  # Use vocab text for reference
    
else:
    raise ValueError(f"Unknown dataset: {CONFIG['dataset']}. Choose 'lotr' or 'fineweb-edu'")

# Create tokenizer based on config
tokenizer_type = CONFIG.get("tokenizer", "char")

if tokenizer_type == "char":
    # Character-level tokenizer
    tokenizer = CharacterTokenizer(vocab_text if CONFIG["dataset"] == "fineweb-edu" else dataset_text)
    print(f"✓ Character tokenizer created")
    print(f"  Vocabulary size: {tokenizer.vocab_size}")
    print(f"  First 50 chars: {''.join(tokenizer.vocab[:50])}...")
    
elif tokenizer_type == "gpt2":
    # GPT-2 BPE tokenizer via tiktoken
    print(f"Loading GPT-2 tokenizer via tiktoken...")
    tokenizer = TiktokenWrapper("gpt2")
    print(f"✓ GPT-2 tokenizer created")
    print(f"  Vocabulary size: {tokenizer.vocab_size:,}")
    print(f"  Encoding: tiktoken (fast BPE)")
    
else:
    raise ValueError(f"Unknown tokenizer: {tokenizer_type}. Choose 'char' or 'gpt2'")

CONFIG["vocab_size"] = tokenizer.vocab_size

# Test tokenizer interface
print(f"\nTokenizer interface test:")
test_text = "hello world"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"  encode('{test_text}') = {encoded}")
print(f"  decode([...]) = '{decoded}'")

Using streaming mode - will tokenize on-the-fly
Loading GPT-2 tokenizer via tiktoken...
✓ GPT-2 tokenizer created
  Vocabulary size: 50,257
  Encoding: tiktoken (fast BPE)

Tokenizer interface test:
  encode('hello world') = [31373, 995]
  decode([...]) = 'hello world'


### Load and Prepare Data with HuggingFace Datasets

Use the tokenizer to create a dataset of fixed-length sequences.

**Two modes supported:**
- **In-memory**: Load a fixed number of documents into memory (faster iteration, limited size)
- **Streaming**: Stream the full dataset and tokenize on-the-fly (unlimited size, memory-efficient)

Set `dataset_subset_size` to `None` or `"all"` to enable streaming mode.

In [5]:
import torch
from datasets import Dataset
from torch.utils.data import IterableDataset

class StreamingTextDataset(IterableDataset):
    """
    Streaming dataset that tokenizes on-the-fly.
    Reads from HuggingFace streaming dataset and creates fixed-length chunks.
    """
    
    def __init__(self, dataset_name, dataset_config, split, tokenizer, block_size, train_split=0.8, is_train=True, seed=42):
        self.dataset_name = dataset_name
        self.dataset_config = dataset_config
        self.split = split
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.train_split = train_split
        self.is_train = is_train
        self.seed = seed
        
    def __iter__(self):
        from datasets import load_dataset
        import random
        
        # Load streaming dataset
        ds = load_dataset(
            self.dataset_name,
            name=self.dataset_config,
            split=self.split,
            streaming=True
        )
        
        # Buffer for accumulating tokens
        token_buffer = []
        chunk_size = self.block_size + 1
        
        # Use hash-based splitting for deterministic train/val split
        random.seed(self.seed)
        
        for idx, doc in enumerate(ds):
            # Deterministic split: use document index hash
            doc_hash = hash(f"{self.seed}_{idx}") % 100
            is_train_doc = (doc_hash < int(self.train_split * 100))
            
            # Skip if document doesn't belong to this split
            if is_train_doc != self.is_train:
                continue
            
            # Tokenize and add to buffer (no lowercasing for BPE tokenizers)
            text = doc["text"]
            tokens = self.tokenizer.encode(text)
            token_buffer.extend(tokens)
            
            # Yield chunks when buffer is large enough
            while len(token_buffer) >= chunk_size:
                chunk = token_buffer[:chunk_size]
                token_buffer = token_buffer[chunk_size:]
                yield {"input_ids": chunk}

# Check if using streaming mode
if CONFIG.get("_use_streaming", False):
    print("Creating streaming datasets...")
    
    # Create streaming datasets
    train_dataset = StreamingTextDataset(
        dataset_name="HuggingFaceFW/fineweb-edu",
        dataset_config="sample-10BT",
        split="train",
        tokenizer=tokenizer,
        block_size=CONFIG["block_size"],
        train_split=CONFIG["train_split"],
        is_train=True,
        seed=CONFIG["seed"]
    )
    
    val_dataset = StreamingTextDataset(
        dataset_name="HuggingFaceFW/fineweb-edu",
        dataset_config="sample-10BT",
        split="train",
        tokenizer=tokenizer,
        block_size=CONFIG["block_size"],
        train_split=CONFIG["train_split"],
        is_train=False,
        seed=CONFIG["seed"]
    )
    
    print("Streaming datasets created (will tokenize on-the-fly)")
    print(f"Using {int(CONFIG['train_split']*100)}% for training, {int((1-CONFIG['train_split'])*100)}% for validation")
    
    # Store datasets for later use
    split_dataset = {"train": train_dataset, "test": val_dataset}
    
else:
    # Original in-memory approach
    print("Creating in-memory dataset...")
    
    # Tokenize entire text
    tokens = tokenizer.encode(dataset_text)

    # Split into chunks of block_size + 1 (for input and target)
    chunk_size = CONFIG["block_size"] + 1
    n_chunks = len(tokens) // chunk_size
    chunks = [
        tokens[i * chunk_size : (i + 1) * chunk_size]
        for i in range(n_chunks)
    ]

    # Create dataset from chunks (using "input_ids" to match transformers convention)
    dataset = Dataset.from_dict({"input_ids": chunks})

    # Split into train and validation
    split_dataset = dataset.train_test_split(
        test_size=1 - CONFIG["train_split"],
        seed=CONFIG["seed"]
    )

    print(f"Total chunks: {len(dataset):,}")
    print(f"Train: {len(split_dataset['train']):,}")
    print(f"Validation: {len(split_dataset['test']):,}")

    # Show a sample
    sample = split_dataset["train"][0]["input_ids"]
    print(f"\nSample ({len(sample)} tokens):")
    print(f"  Input:  '{tokenizer.decode(sample[:-1])[:100]}...'")
    print(f"  Target: '{tokenizer.decode(sample[1:])[:100]}...'")

print(f"\nDataset uses 'input_ids' key to match HuggingFace conventions!")

Creating streaming datasets...
Streaming datasets created (will tokenize on-the-fly)
Using 80% for training, 19% for validation

Dataset uses 'input_ids' key to match HuggingFace conventions!


### Create DataLoaders

Convert the HuggingFace datasets to PyTorch DataLoaders. Settings are automatically optimized based on:
- **Hardware**: CUDA uses more workers and prefetching; MPS uses 0 workers (avoids multiprocessing issues)
- **Dataset mode**: Streaming uses different worker counts than in-memory
- **Compatibility**: `persistent_workers` only enabled when `num_workers > 0`

In [6]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    """Convert batch of token sequences to input/target pairs."""
    tokens = torch.tensor([item["input_ids"] for item in batch])
    x = tokens[:, :-1]  # Input: all but last
    y = tokens[:, 1:]   # Target: all but first (shifted by 1)
    return x, y

# Check if using streaming mode
is_streaming = CONFIG.get("_use_streaming", False)

# Get auto-detected dataloader settings
num_workers = CONFIG.get("num_workers", 4)
persistent_workers = CONFIG.get("persistent_workers", False)
prefetch_factor = CONFIG.get("prefetch_factor", 2)
pin_memory = CONFIG.get("pin_memory", True)

# Prepare kwargs (exclude None values)
train_kwargs = {
    "batch_size": CONFIG["batch_size"],
    "collate_fn": collate_fn,
    "num_workers": num_workers,
    "pin_memory": pin_memory,
}

val_kwargs = train_kwargs.copy()

# Add optional parameters only if applicable
if num_workers > 0:
    if persistent_workers:
        train_kwargs["persistent_workers"] = True
        val_kwargs["persistent_workers"] = True
    if prefetch_factor is not None:
        train_kwargs["prefetch_factor"] = prefetch_factor
        val_kwargs["prefetch_factor"] = prefetch_factor

# Shuffle only for in-memory datasets
if not is_streaming:
    train_kwargs["shuffle"] = True

# Create dataloaders
print(f"Creating DataLoaders ({'streaming' if is_streaming else 'in-memory'} mode)...")
print(f"  Workers: {num_workers}, Persistent: {persistent_workers}, Prefetch: {prefetch_factor}, Pin memory: {pin_memory}")

train_loader = DataLoader(split_dataset["train"], **train_kwargs)
val_loader = DataLoader(split_dataset["test"], **val_kwargs)

# Check batch shapes (only for in-memory)
if not is_streaming:
    x_batch, y_batch = next(iter(train_loader))
    print(f"  Batch shape: {x_batch.shape}")

print(f"✓ DataLoaders created!")

Creating DataLoaders (streaming mode)...
  Workers: 4, Persistent: True, Prefetch: 4, Pin memory: True
✓ DataLoaders created!


## 4. Mamba Model Implementation

**This is the core!** The following cells contain the exact same Mamba implementation from the original notebook. Focus your attention here - this is what makes Mamba work.

### MambaBlock: The Selective SSM Core

This block implements the selective state space mechanism with data-dependent parameters.

In [7]:
import torch.nn as nn
import torch.nn.functional as F

class MambaBlock(nn.Module):
    """A single Mamba block with selective SSM."""
    
    def __init__(
        self, 
        d_model, 
        d_state=16, 
        expand=2, 
        conv_kernel=4,
        ssm_backend="cuda"
    ):
        super().__init__()

        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = d_model * expand  # Expanded dimension
        self.conv_kernel = conv_kernel
        self.ssm_backend = ssm_backend
                
        # 1. Input projection (to 2*d_inner for x and z)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        
        # 2. Convolution (adds local context within sequence)
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=conv_kernel,
            padding=conv_kernel - 1,
            groups=self.d_inner  # Depthwise convolution
        )
        
        # 3. SSM parameters
        # A: Transition matrix (S4D-Lin initialization)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))  # Log space for stability
        
        # D: Skip connection parameter (used by CUDA implementation)
        self.D = nn.Parameter(torch.ones(self.d_inner))
        
        # 4. Python version: Projection to compute B, C, Δ (scalar delta)
        self.x_proj = nn.Linear(
            self.d_inner,
            d_state * 2 + 1,  # B (d_state) + C (d_state) + Δ (1)
            bias=False
        )
        
        # 5. CUDA version: Separate projections for per-channel delta
        self.dt_proj = nn.Linear(self.d_inner, self.d_inner, bias=True)
        self.B_proj = nn.Linear(self.d_inner, d_state, bias=False)
        self.C_proj = nn.Linear(self.d_inner, d_state, bias=False)
        
        # 6. Output projection
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
    
    def _preprocess_common(self, x):
        """
        Shared preprocessing: input projection, conv, activation.
        
        Returns:
            x_conv: (B, T, d_inner) - convolved and activated input
            z: (B, T, d_inner) - gate values
        """
        B, T, _ = x.shape
        
        # Input projection and split
        x_and_z = self.in_proj(x)  # (B, T, 2*d_inner)
        x, z = x_and_z.chunk(2, dim=-1)  # Each: (B, T, d_inner)
        
        # Depthwise convolution
        x_conv = x.transpose(1, 2)  # (B, d_inner, T)
        x_conv = self.conv1d(x_conv)[:, :, :T]  # (B, d_inner, T)
        x_conv = x_conv.transpose(1, 2)  # (B, T, d_inner)
        x_conv = F.silu(x_conv)
        
        return x_conv, z
    
    def forward(self, x, states=None):
        """
        Forward pass with selectable backend (Python or CUDA).

        Args:
            x: (B, T, d_model) input tensor
            states: Optional previous hidden states (B, I, S). If None, starts from zeros.

        Returns:
            output: (B, T, d_model) output tensor
            new_states: (B, I, S) final hidden states
        """
        
        if self.ssm_backend == "cuda": return self._forward_cuda(x, states)
        else: return self._forward_python(x, states)
    
    def _forward_cuda(self, x, states=None):
        """
        CUDA-optimized forward pass using mamba-ssm's selective_scan_fn.
        
        This is 30-100x faster than the Python implementation due to:
        - Parallel scan algorithm (vs sequential loop)
        - Fused CUDA kernels
        - Optimized memory access patterns
        """
        
        from mamba_ssm.ops.selective_scan_interface import selective_scan_fn

        # Common preprocessing
        x_conv, z = self._preprocess_common(x)
        
        # CUDA-specific: Compute per-channel delta from x_conv
        delta = self.dt_proj(x_conv)  # (B, T, d_inner)
        delta = F.softplus(delta)  # Ensure positive
        delta = delta.to(dtype=x_conv.dtype)  # Match input dtype for CUDA kernel
        
        # CUDA-specific: Compute B and C from x_conv
        B_param = self.B_proj(x_conv)  # (B, T, d_state)
        C_param = self.C_proj(x_conv)  # (B, T, d_state)
        
        # Get transition matrix (A and D must be float32 for CUDA kernel)
        A = -torch.exp(self.A_log.float())  # (d_inner, d_state) - float32 required
        D = self.D.float()  # float32 required
        
        # Call optimized CUDA kernel (expects channels-first: B, d_inner, T)
        y = selective_scan_fn(
            x_conv.transpose(1, 2).contiguous(),  # (B, d_inner, T)
            delta.transpose(1, 2).contiguous(),   # (B, d_inner, T)
            A,                                     # (d_inner, d_state) - float32
            B_param.transpose(1, 2).contiguous(),  # (B, d_state, T)
            C_param.transpose(1, 2).contiguous(),  # (B, d_state, T)
            D,                                     # float32
            z=z.transpose(1, 2).contiguous(),      # (B, d_inner, T)
            delta_bias=None,
            delta_softplus=False,                  # Already applied
            return_last_state=True
        )
        
        # Unpack results
        y_out, final_state = y
        y_out = y_out.transpose(1, 2)  # (B, T, d_inner)
        
        # Output projection (gating already applied in kernel)
        return self.out_proj(y_out), final_state
    
    def _forward_python(self, x, states=None):
        """
        Pure Python implementation with explicit SSM recurrence.
        
        EDUCATIONAL VERSION: Slower but shows exactly how the selective SSM works.
        """
        
        # Common preprocessing
        x_conv, z = self._preprocess_common(x)
        
        B, T, _ = x.shape
        I = self.d_inner
        S = self.d_state
        
        # Python-specific: Compute scalar delta, B, C from x_proj
        ssm_params = self.x_proj(x_conv)  # (B, T, 2*d_state + 1)
        delta, B_param, C_param = torch.split(
            ssm_params, [1, S, S], dim=-1
        )
        delta = F.softplus(delta)  # (B, T, 1) - ensure positive
        delta = delta.expand(-1, -1, I)  # (B, T, d_inner) - broadcast across channels
        
        # Get transition matrix
        A = -torch.exp(self.A_log)  # (I, S)
        
        # Initialize state and output
        h = torch.zeros(B, I, S, device=x.device, dtype=x.dtype) if states is None else states
        y = torch.zeros(B, T, I, device=x.device, dtype=x.dtype)
        
        # ------------------------------------------------------------
        # Selective scan: SSM recurrence over time (THE CORE LOOP)
        # ------------------------------------------------------------
        for t in range(T):
            # Get timestep-specific values
            delta_t = delta[:, t, :].unsqueeze(-1)  # (B, I, 1)
            x_t = x_conv[:, t].unsqueeze(-1)        # (B, I, 1)
            B_t = B_param[:, t].unsqueeze(1)        # (B, 1, S)
            C_t = C_param[:, t].unsqueeze(1)        # (B, 1, S)
            
            # Discretize: A_discrete = exp(A * Δ_t)
            A_disc = torch.exp(A.unsqueeze(0) * delta_t)  # (B, I, S)
            
            # State update: h_t = A_discrete ⊙ h_{t-1} + B_t ⊙ x_t
            h = A_disc * h + B_t * x_t  # (B, I, S)
            
            # Output projection: y_t = C_t · h_t
            y[:, t] = torch.sum(C_t * h, dim=-1)  # (B, I)
        
        # Apply gating
        y = y * F.silu(z)  # (B, T, I)
        
        # Output projection
        return self.out_proj(y), h

### ResidualMambaBlock: Normalize and Add

In [8]:
class ResidualMambaBlock(nn.Module):
    """Mamba block with pre-norm and residual connection."""
    
    def __init__(self, d_model, d_state=16, expand=2, conv_kernel=4, ssm_backend="cuda"):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MambaBlock(d_model, d_state, expand, conv_kernel, ssm_backend)
    
    def forward(self, x):
        # Pre-norm with residual: x + Mamba(LayerNorm(x))
        # MambaBlock returns (output, states), we only need the output
        output, _ = self.mamba(self.norm(x))
        return x + output

### Full Mamba Model: Stack Layers and Add Embeddings

The complete language model with token embeddings and output head.

In [9]:
class Mamba(nn.Module):
    """Full Mamba language model."""
    
    def __init__(
        self,
        vocab_size,
        n_embed,
        d_model,
        d_state,
        n_layers,
        expand,
        conv_kernel,
        ssm_backend="cuda",
    ):
        super().__init__()
        
        # Token embeddings
        self.embeddings = nn.Embedding(vocab_size, n_embed)
        
        # Project to model dimension if needed
        self.embed_proj = (
            nn.Linear(n_embed, d_model, bias=False)
            if n_embed != d_model
            else nn.Identity()
        )
        
        # Stack of Mamba blocks
        self.blocks = nn.ModuleList([
            ResidualMambaBlock(d_model, d_state, expand, conv_kernel, ssm_backend)
            for _ in range(n_layers)
        ])
        
        # Output head
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len) token indices
        Returns:
            logits: (batch, seq_len, vocab_size)
        """
        # Embed tokens
        x = self.embeddings(x)  # (B, T) -> (B, T, C)
        x = self.embed_proj(x)  # (B, T, C) -> (B, T, H)
        
        # Pass through Mamba blocks
        for block in self.blocks:
            x = block(x)
        
        # Output projection
        x = self.ln_out(x)
        logits = self.head(x)  # (batch, seq_len, vocab_size)
        
        return logits

print("Mamba model implementation complete!")
print("\nThese ~200 lines contain both Python and CUDA implementations.")
print("Everything else is just data loading and training boilerplate.")

Mamba model implementation complete!

These ~200 lines contain both Python and CUDA implementations.
Everything else is just data loading and training boilerplate.


## 5. PyTorch Lightning Wrapper

Wrap the Mamba model in a Lightning module to handle training, validation, and optimization. This replaces ~30 cells of training loop code!

In [10]:
import lightning as L
import time

class MambaLightning(L.LightningModule):
    """Lightning wrapper for Mamba model."""
    
    def __init__(self, config):
        super().__init__()
        self.save_hyperparameters(config)
        
        # Create the Mamba model
        self.model = Mamba(
            vocab_size=config["vocab_size"],
            n_embed=config["n_embed"],
            d_model=config["hidden_size"],
            d_state=config["state_size"],
            n_layers=config["n_layers"],
            expand=config["expand_factor"],
            conv_kernel=config["conv_kernel"],
            ssm_backend=config.get("ssm_backend", "python"),
        )
        
        self.criterion = nn.CrossEntropyLoss()
        
        # Track tokens/sec
        self.last_step_time = None
        self.tokens_per_sec = 0
    
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        
        # Start timing
        step_start = time.perf_counter()
        
        logits = self(x)
        loss = self.criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        
        # Calculate tokens/sec
        step_end = time.perf_counter()
        step_time = step_end - step_start
        
        if step_time > 0:
            batch_size, seq_len = x.shape
            tokens_processed = batch_size * seq_len
            self.tokens_per_sec = tokens_processed / step_time
        
        self.log("train_loss", loss, prog_bar=True)
        self.log("tokens_per_sec", self.tokens_per_sec, prog_bar=True, logger=False)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        perplexity = torch.exp(loss)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_perplexity", perplexity, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        # Use fused AdamW on CUDA for ~5-10% speedup
        use_fused = self.hparams.get("use_fused_adamw", False) and torch.cuda.is_available()
        
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.learning_rate,
            fused=use_fused
        )
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.hparams.max_epochs,
            eta_min=self.hparams.get("min_lr", 1e-6)
        )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1,
            },
        }
        
print("Lightning wrapper complete!")
print("This ~30 line class replaces all the training loop boilerplate.")

Lightning wrapper complete!
This ~30 line class replaces all the training loop boilerplate.


## 6. Train the Model

PyTorch Lightning handles everything: device management, progress bars, logging, checkpointing, and early stopping.

In [11]:
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

# Create model
model = MambaLightning(CONFIG)

# Apply torch.compile for 30-40% speedup (requires PyTorch 2.0+)
compile_mode = CONFIG.get("torch_compile")
if compile_mode:
    print(f"Compiling model with torch.compile (mode={compile_mode})...")
    model.model = torch.compile(model.model, mode=compile_mode)
    print("✓ Model compiled!")

# Create logger
logger = CSVLogger("logs", name="mamba_minimal")

# Create checkpoint callback to save best model
# (litmodels package enables automatic cloud sync features)
checkpoint_callback = ModelCheckpoint(
    dirpath=CONFIG["checkpoint_dir"],
    filename="mamba-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,  # Only keep the best checkpoint
    save_last=True,  # Also save the last checkpoint
    verbose=True,
)

# Create early stopping callback
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=CONFIG["early_stop_patience"],
    mode="min",
    verbose=False,
)

# Create trainer with auto-detected settings
trainer = L.Trainer(
    max_epochs=CONFIG["max_epochs"],
    accelerator="auto",  # Auto-detect accelerator
    devices=1,
    precision=CONFIG["precision"],  # Use auto-detected precision (no AMP on MPS)
    logger=logger,
    callbacks=[early_stop, checkpoint_callback],
    enable_progress_bar=True,
)

# Train (with optional resume from checkpoint)
print("\nStarting training...")
print(f"Device: {CONFIG['device']}")
print(f"Precision: {CONFIG['precision']}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"Torch compile: {'✓ (' + CONFIG.get('torch_compile') + ')' if CONFIG.get('torch_compile') else '✗'}")
print(f"Fused AdamW: {'✓' if CONFIG.get('use_fused_adamw', False) else '✗'}")

if CONFIG["resume_from_checkpoint"]:
    print(f"\nResuming from checkpoint: {CONFIG['resume_from_checkpoint']}")
    trainer.fit(model, train_loader, val_loader, ckpt_path=CONFIG["resume_from_checkpoint"])
else:
    print("\nTraining from scratch...")
    trainer.fit(model, train_loader, val_loader)

print("\nTraining complete!")
print(f"Best checkpoint saved at: {checkpoint_callback.best_model_path}")
print(f"Best validation loss: {checkpoint_callback.best_model_score:.4f}")

Compiling model with torch.compile (mode=max-autotune)...


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:751: Checkpoint directory /home/tsilva/repos/tsilva/aiml-notebooks/notebooks/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


✓ Model compiled!

Starting training...
Device: cuda
Precision: bf16-mixed
Batch size: 24
Torch compile: ✓ (max-autotune)
Fused AdamW: ✓

Training from scratch...


/home/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | OptimizedModule  | 85.0 M | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
85.0 M    Trainable params
0         Non-trainable params
85.0 M    Total params
339.993   Total estimated model params size (MB)
128       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.13/site-packages/torch/_dynamo/variables/functions.py:1598: UserWarning: Dynamo does not know how to trace the builtin `selective_scan_cuda.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1.fwd.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround.
If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use `torch.compiler.allow_in_graph`.
  torch._dynamo.utils.warn_once(explanation + "\n" + "\n".join(hints))
Autotune Choices Stats:
{"num_choices": 16, "num_triton_choices": 15, "best_kernel": "triton_bmm_24", "be

Training: |          | 0/? [00:00<?, ?it/s]

Autotune Choices Stats:
{"num_choices": 16, "num_triton_choices": 15, "best_kernel": "triton_mm_99", "best_kernel_desc": "ACC_TYPE='tl.float32', ALLOW_TF32=True, BLOCK_K=64, BLOCK_M=64, BLOCK_N=128, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=3, num_warps=4", "best_time": 0.2959359884262085, "best_triton_pos": 0}
AUTOTUNE mm(24576x1024, 1024x1024)
strides: [1024, 1], [1, 1024]
dtypes: torch.bfloat16, torch.bfloat16
  triton_mm_99 0.2959 ms 100.0% ACC_TYPE='tl.float32', ALLOW_TF32=True, BLOCK_K=64, BLOCK_M=64, BLOCK_N=128, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=3, num_warps=4
  triton_mm_103 0.2996 ms 98.8% ACC_TYPE='tl.float32', ALLOW_TF32=True, BLOCK_K=32, BLOCK_M=128, BLOCK_N=128, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=3, num_warps=4
  triton_mm_97 0.3031 ms 97.6% ACC_TYPE='tl.float32', ALLOW_TF32=True, BLOCK_K=32, BLOCK_M=64, BLOCK_N=128, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=3, num_warps=4
  triton_mm_100 0.3052 ms

SystemExit: 1

/home/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Upload Best Checkpoint to Lightning Model Registry

If enabled, upload the best checkpoint to the Lightning model registry for versioning and sharing.

In [ ]:
from litmodels import upload_model
import os

# Upload best checkpoint to Lightning model registry if enabled
if CONFIG["upload_to_litmodels"] and CONFIG["litmodels_name"]:
    print("Uploading best checkpoint to Lightning model registry...")
    
    # Create metadata with training info
    metadata = {
        "dataset": CONFIG["dataset"],
        "vocab_size": str(CONFIG["vocab_size"]),
        "n_layers": str(CONFIG["n_layers"]),
        "hidden_size": str(CONFIG["hidden_size"]),
        "state_size": str(CONFIG["state_size"]),
        "best_val_loss": f"{checkpoint_callback.best_model_score:.4f}",
        "framework": "pytorch-lightning",
        "model_type": "mamba",
    }
    
    # Upload the best checkpoint
    result = upload_model(
        name=CONFIG["litmodels_name"],
        model=checkpoint_callback.best_model_path,
        metadata=metadata,
        verbose=True,
    )
    
    print(f"✓ Model uploaded successfully!")
    print(f"  Model ID: {result.model_id}")
    print(f"  Version: {result.version}")
    print(f"  URL: {result.url}")
    
elif CONFIG["upload_to_litmodels"] and not CONFIG["litmodels_name"]:
    print("⚠️  upload_to_litmodels is True but litmodels_name is not set!")
    print("   Set CONFIG['litmodels_name'] = 'username/teamspace/modelname'")
    
else:
    print("Skipping upload to Lightning model registry (upload_to_litmodels=False)")
    print("To enable: Set CONFIG['upload_to_litmodels'] = True and CONFIG['litmodels_name'] = 'username/teamspace/modelname'")

### Load Best Checkpoint

After training, the best model (lowest validation loss) is automatically saved. Load it for inference or further training.

In [ ]:
# Load the best checkpoint for inference or continued training
best_model = MambaLightning.load_from_checkpoint(
    checkpoint_callback.best_model_path,
    config=CONFIG  # Pass the config to the model
)

print(f"Loaded best model from: {checkpoint_callback.best_model_path}")
print(f"Best validation loss: {checkpoint_callback.best_model_score:.4f}")

# To resume training from this checkpoint later, set:
# CONFIG["resume_from_checkpoint"] = checkpoint_callback.best_model_path

## 7. Plot Training Curves

Visualize how the model learned over time.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Read metrics from CSV logger
metrics = pd.read_csv(f"{logger.log_dir}/metrics.csv")

# Aggregate by epoch
train_metrics = metrics[["epoch", "train_loss"]].dropna()
val_metrics = metrics[["epoch", "val_loss"]].dropna()
train_metrics = train_metrics.groupby("epoch").mean().reset_index()
val_metrics = val_metrics.groupby("epoch").mean().reset_index()

# Plot
plt.figure(figsize=(12, 6))
plt.plot(
    train_metrics["epoch"],
    train_metrics["train_loss"],
    label="Train Loss",
    marker="o",
    linewidth=2,
    color="#4ECDC4",
)
plt.plot(
    val_metrics["epoch"],
    val_metrics["val_loss"],
    label="Validation Loss",
    marker="s",
    linewidth=2,
    color="#FF6B6B",
)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("Mamba Training Progress", fontsize=14, fontweight="bold")
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Report statistics
best_val_loss = val_metrics["val_loss"].min()
best_epoch = val_metrics.loc[val_metrics["val_loss"].idxmin(), "epoch"]
print(f"Best validation loss: {best_val_loss:.4f} at epoch {int(best_epoch)}")
print(f"Final train loss: {train_metrics['train_loss'].iloc[-1]:.4f}")
print(f"Final val loss: {val_metrics['val_loss'].iloc[-1]:.4f}")

## 8. Text Generation

Test the trained model by generating text continuations.

In [12]:
import time
import torch
import torch.nn.functional as F

def generate(model, prompt, max_len=CONFIG["block_size"], temperature=1.0):
    """
    Generate text continuation from a prompt.
    
    Args:
        model: The trained model
        prompt: Starting text
        max_len: Maximum total length
        temperature: Sampling temperature (lower = more conservative)
    """
    # Ensure model is on the correct device and in eval mode
    device = torch.device(CONFIG["device"])
    model = model.to(device)
    model.eval()

    tokens = tokenizer.encode(prompt)
    max_new = max_len - len(tokens)

    print(prompt, end="", flush=True)

    def sample(logits):
        logits = logits[:, -1] / temperature
        probs = F.softmax(logits, dim=-1)
        return torch.multinomial(probs, 1).item()

    start_time = time.perf_counter()
    generated = 0

    with torch.no_grad():
        for _ in range(max_new):
            # Convert current tokens to tensor
            x = torch.tensor([tokens], device=device)
            
            # Get logits from model
            logits = model(x)
            
            # Sample next token
            t = sample(logits)
            tokens.append(t)
            generated += 1
            print(tokenizer.decode([t]), end="", flush=True)

    elapsed = time.perf_counter() - start_time
    tps = generated / elapsed if elapsed > 0 else 0.0

    print(f"\n\nGenerated {generated} tokens in {elapsed:.2f}s ({tps:.2f} tokens/sec)")

print("Generation function ready!")

Generation function ready!


### Generate Samples

Try different prompts and temperatures to see what the model learned.

In [13]:
print("Sample 1 (temperature=1.0):")
generate(model, "frodo picked up the ", temperature=1.0)

Sample 1 (temperature=1.0):
frodo picked up the MP to be handled in spiritual norms, and had suggested that some individuals who have picked up human sea cell could accomplish Continent (TCually, divided and acknowledged) taken from a casualties of inequarians.
We have a high school of 20 percent be

CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 51 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.
CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 51 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.
CUDAGraph supports dynamic shapes by recording a new graph

 close to gender infections that are thought to be charges of. Guong Tyulness Kimville and Scotland reads at affopy: Proceedings of four factors in the United States. In 2008, researchers��.
Where - Gokminishes about Tennessee and civil literature, sealing electrons, and Pharaoh stories, exploring this new related disorder that both industries and researchers.
One illustational Beerfully works 2008,

KeyboardInterrupt: 

In [ ]:
print("\nSample 2 (temperature=1.0):")
generate(model, "gandalf said ", temperature=1.0)

In [ ]:
print("\nSample 3 (temperature=0.7 - more focused):")
generate(model, "the ring of power ", temperature=0.7)

In [ ]:
print("\nSample 4 (temperature=0.7 - test length generalization):")
generate(model, "the ring of power ", temperature=0.7, max_len=CONFIG["block_size"] * 10)

## 9. Key Takeaways

### What We Accomplished

We built the **exact same Mamba model** as the detailed introduction, but with:
- **~60% fewer cells** (15 vs 65 cells)
- **Modern best practices** (HuggingFace datasets, PyTorch Lightning)
- **Focus on what matters**: The Mamba implementation itself

### The Core Mamba Code

Only ~150 lines of code in 3 classes:
1. **MambaBlock**: Selective SSM with data-dependent parameters (B, C, Δ)
2. **ResidualMambaBlock**: Normalization + residual connections
3. **Mamba**: Stack layers and add embeddings

### What Libraries Handled

**Custom CharacterTokenizer** (~40 lines):
- Transformers-compatible interface (encode, decode, __call__)
- Easy to swap with any HuggingFace PreTrainedTokenizer
- Uses "input_ids" key to match HuggingFace conventions

**HuggingFace Datasets** (~3 cells vs ~20 cells):
- Data loading and tokenization
- Train/validation splitting
- Batching and shuffling

**PyTorch Lightning** (~2 cells vs ~15 cells):
- Device management
- Training/validation loops
- Progress bars and logging
- Early stopping
- Checkpointing

### Swapping Tokenizers

To use a different tokenizer, simply replace:
```python
tokenizer = CharacterTokenizer(text)
```

With any HuggingFace tokenizer:
```python
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
```

The rest of the code remains unchanged!

### The Selective SSM Mechanism

The key innovation in Mamba:
1. **Input-dependent parameters**: B, C, Δ computed from input
2. **Discretization**: Convert continuous SSM to discrete time steps
3. **Selective scan**: Recurrent computation with changing parameters
4. **Linear complexity**: O(T) vs O(T²) for Transformers

### Next Steps

To deepen your understanding:
1. **Compare notebooks**: Read the detailed version to understand SSM theory
2. **Modify the config**: Try different state sizes, expand factors, etc.
3. **Visualize parameters**: Plot the learned Δ, B, C values
4. **Swap tokenizers**: Try GPT2Tokenizer or other HuggingFace tokenizers
5. **Scale up**: Increase model size and training data
6. **Try other tasks**: Apply Mamba to different sequence modeling problems

### Why This Matters

Mamba shows that:
- **Selectivity** (content-based parameter adjustment) is powerful
- **Linear complexity** doesn't sacrifice expressiveness
- **Classic control theory** can inspire modern deep learning

The minimal boilerplate approach lets you focus on these core insights instead of data pipelines and training loops!